<a href="https://colab.research.google.com/github/fanat503/Induction-Heads-Tinystories/blob/main/Tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

def load_checkpoint(path, model, device):
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    return model


def make_repeated_sequence(tokenizer, length=50, device='cuda'):
    random_tokens = torch.randint(100, 5000, (length,))
    repeated = torch.cat([random_tokens, random_tokens])
    return repeated.unsqueeze(0).to(device)


def collect_attention_all_layers(model, input_ids):
    attention_maps = {}

    with torch.no_grad():
        x = model.transformer.wte(input_ids) + \
            model.transformer.wpe(
                torch.arange(input_ids.size(1), device=input_ids.device)
            )

        for layer_idx, block in enumerate(model.transformer.h):
            x_norm = block.ln_1(x)

            attn_out, attn_weights = block.attn.forward_with_attention(x_norm)

            attention_maps[layer_idx] = attn_weights.cpu()
            x = x + attn_out
            x = x + block.mlp(block.ln_2(x))

    return attention_maps

In [ ]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import time


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        y = F.scaled_dot_product_attention(q,k,v, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        y = self.c_proj(y)
        return y

    def forward_with_attention(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)

        return y, att


class MLP(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
    self.c_proj.NANOGPT_SCALE_INIT = 1

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x


class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = MLP(config)

  def forward(self, x):
    x = x + self.attn(self.ln_1(x))
    x = x + self.mlp(self.ln_2(x))
    return x


@dataclass
class GPTConfig:
    # --- АРХИТЕКТУРА (GPT-2 124M) ---
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1

grad_accum_steps = 32

class GPT(nn.Module):

  def __init__(self, config):
    super().__init__()
    self.config = config

    self.transformer = nn.ModuleDict(dict(
      wte = nn.Embedding(config.vocab_size, config.n_embd),
      wpe = nn.Embedding(config.block_size, config.n_embd),
      h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
      ln_f = nn.LayerNorm(config.n_embd),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    self.transformer.wte.weight = self.lm_head.weight

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      std = 0.02
      if hasattr(module, 'NANOGPT_SCALE_INIT'):
        std *= (2 * self.config.n_layer) ** -0.5
      torch.nn.init.normal_(module.weight, mean = 0.0, std = std)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


  def forward(self,idx, targets=None):
    B,T = idx.size()
    assert T <= self.config.block_size
    pos = torch.arange(0, T, dtype = torch.long, device=idx.device)
    pos_emb = self.transformer.wpe(pos)
    tok_emb = self.transformer.wte(idx)
    x = tok_emb + pos_emb
    for block in self.transformer.h:
      x = block(x)

    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)
    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
    return logits, loss

In [ ]:
def compute_induction_score(attention_maps, seq_len_half):
    scores = {}
    for layer in attention_maps:
        for head in range(12):
            ind_pat = []
            for t in range(seq_len_half, seq_len_half * 2):
                source = t - seq_len_half + 1
                ind_pat.append(
                    attention_maps[layer][0, head, t, source]
                )
            ind_score = torch.tensor(ind_pat).mean()
            scores[(layer, head)] = ind_score
    return scores


def compute_previous_token_score(attention_maps):
    Score = {}
    for layer in attention_maps:
      for head in range(12):
        prev_tok_pat = []
        for t in range(1, 100):
            prev_tok_pat.append(attention_maps[layer][0, head, t, t-1])
        Score[(layer, head)] = torch.tensor(prev_tok_pat).mean()
    return Score

In [ ]:
random_tokens = torch.randint(100, 5000, (50,))
repeated = torch.cat([random_tokens, random_tokens])
input_ids = repeated.unsqueeze(0).to(device)

attention_maps = collect_attention_all_layers(model, input_ids)

scores = compute_induction_score(attention_maps, seq_len_half=50)

for key, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"Layer {key[0]:2d}, Head {key[1]:2d}: {score:.4f}")

In [ ]:
random_tokens = torch.randint(100, 5000, (50,))
repeated = torch.cat([random_tokens, random_tokens])
input_ids = repeated.unsqueeze(0).to(device)

attention_maps = collect_attention_all_layers(model, input_ids)


scores = compute_previous_token_score(attention_maps)

for key, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"Layer {key[0]:2d}, Head {key[1]:2d}: {score:.4f}")

In [ ]:
def analyze_checkpoint(path, seq_len_half, model, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    step = checkpoint['step']
    torch.manual_seed(42)

    random_tokens = torch.randint(100, 5000, (seq_len_half,))
    repeated = torch.cat([random_tokens, random_tokens])
    input_ids = repeated.unsqueeze(0).to(device)
    attention_maps = collect_attention_all_layers(model, input_ids)

    ih_scores = compute_induction_score(attention_maps, seq_len_half)
    pth_scores = compute_previous_token_score(attention_maps)
    def print_top_scores(scores, name, top_n=10):
      sorted_scores = sorted(
        scores.items(),
        key=lambda x: x[1].item(),
        reverse=True
        )
      for i, (key, score) in enumerate(sorted_scores[:top_n]):
        print(f"  Layer {key[0]:2d}, Head {key[1]:2d}: {score:.4f}")
      print(f"{'='*40}")

    result = {
        'induction': ih_scores,
        'previous_token': pth_scores,
        'step': step
    }

    print(f"\n Step {step}")
    print_top_scores(ih_scores, "Induction Score")
    print_top_scores(pth_scores, "Previous Token Score")

    return result

import matplotlib.pyplot as plt
import numpy as np

def visualisation(scores, n_layer, n_head, title, step):
  grid = np.zeros((n_layer, n_layer))
  for layer in range(n_layer):
    for head in range(n_head):
      grid[layer, head] = scores[layer, head]
  plt.imshow(grid)
  plt.colorbar(cmap='hot')
  plt.xlabel("Head")
  plt.ylabel("Layer")
  plt.title(title)
  plt.xticks(range(n_head))
  plt.yticks(range(n_layer))
  plt.savefig(f'*{title}_{step}.png', dpi=300)
  plt.show()

In [ ]:
import os
import re
import pandas as pd

folder = '*'
checkpoint_files = []
for f in os.listdir(folder):
    if 'checkpoint' in f and f.endswith('.pt'):
        match = re.search(r'checkpoint(\d+)', f)
        if match:
            step_num = int(match.group(1))
            checkpoint_files.append((step_num, os.path.join(folder, f)))
checkpoint_files.sort(key=lambda x: x[0])

results = {}
model = GPT(GPTConfig()).to('cpu')
for step, path in checkpoint_files:
    print(f"\nОбработка шага {step}...")
    result = analyze_checkpoint(path, 50, model, 'cpu')
    results[step] = result

    visualisation(result['induction'], 12, 12, 'ih_heatmap', step)
    visualisation(result['previous_token'], 12, 12, 'pth_heatmap', step)

rows = []
for step, res in results.items():
    ih_top = max(res['induction'].items(), key=lambda x: x[1].item())
    pth_top = max(res['previous_token'].items(), key=lambda x: x[1].item())

    rows.append({
        'step': step,
        'ih_top_score': ih_top[1].item(),
        'ih_layer': ih_top[0][0],
        'ih_head': ih_top[0][1],
        'pth_top_score': pth_top[1].item(),
        'pth_layer': pth_top[0][0],
        'pth_head': pth_top[0][1],
    })

df = pd.DataFrame(rows)
print(df)
df.to_csv(f'{folder}/analysis_results.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

steps = [6000, 6800, 8400]
ih_scores = [0.052, 0.048, 0.098]
pth_scores = [0.165, 0.151, 0.164]

fig, ax1 = plt.subplots(figsize=(10, 6))


ax1.plot(steps, ih_scores, 'r-o', label='Induction Score (L6H4)', linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('Induction Score', color='red')
ax1.tick_params(axis='y', labelcolor='red')


ax2 = ax1.twinx()
ax2.plot(steps, pth_scores, 'b-o', label='Previous Token Score (top)', linewidth=2)
ax2.set_ylabel('Previous Token Score', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Induction Head Formation Dynamics — GPT-2 124M on TinyStories')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.grid(True, alpha=0.3)

plt.savefig('*formation_dynamics.png', dpi=300)
plt.show()

In [ ]:
steps = [14000, 15000, 16000, 17000, 19000]
pth_scores = [0.1978, 0.2095, 0.2158, 0.2124, 0.2151]

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()
ax2.plot(steps, pth_scores, 'b-o', label='Previous Token Score (top)', linewidth=2)
ax2.set_ylabel('Previous Token Score', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

plt.title('Previous Token Score — GPT-2 124M on TinyStories')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.grid(True, alpha=0.3)

plt.savefig('*Previous Token Score — GPT-2 124M on TinyStories.png', dpi=300)
plt.show()

In [ ]:
import numpy as np

steps = [14000, 15000, 16000, 17000, 18000, 19000]
base_path = '*'

for step in steps:
    path = f'{base_path}{step}.pt'
    result = analyze_checkpoint(path, 50, GPT(GPTConfig()), 'cpu')

    ih_scores = result['induction']
    prev_scores = result['previous_token']

    max_ih_key = max(ih_scores, key=ih_scores.get)
    max_ih_val = ih_scores[max_ih_key]

    max_prev_key = max(prev_scores, key=prev_scores.get)
    max_prev_val = prev_scores[max_prev_key]

    print(f"Step {step:5d} | "
          f"Max IH: {max_ih_val:.4f} L{max_ih_key[0]}H{max_ih_key[1]} | "
          f"Max Prev: {max_prev_val:.4f} L{max_prev_key[0]}H{max_prev_key[1]}")

In [ ]:
@torch.no_grad()
def test_induction_score(model, seq_len_half=50):
    model.eval()

    pattern = list(range(1, seq_len_half + 1))
    full_seq = pattern + pattern
    x = torch.tensor([full_seq])

    attention_maps = {}

    def make_hook(layer_idx):
        def hook(module, input, output):
            pass
        return hook

    tok_emb = model.transformer.wte(x)
    pos = torch.arange(0, x.shape[1]).unsqueeze(0)
    pos_emb = model.transformer.wpe(pos)
    hidden = tok_emb + pos_emb

    for i, block in enumerate(model.transformer.h):
        attn_out, att = block.attn.forward_with_attention(block.ln_1(hidden))
        attention_maps[i] = att.detach().cpu()

        hidden = hidden + attn_out
        hidden = hidden + block.mlp(block.ln_2(hidden))

    scores = compute_induction_score(attention_maps, seq_len_half)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    for (layer, head), score in sorted_scores[:10]:
        print(f"  Layer {layer:2d}, Head {head:2d}: {score:.4f}")
    print("=" * 40)

    l6h4 = scores.get((6, 4), 0)
    print(f"\nL6H4: {l6h4:.4f}")

    if l6h4 > 0.1:
        print("*")
    elif l6h4 > 0.03:
        print("*")
    else:
        print("*")

    return scores

checkpoint = torch.load(
    '*',
    map_location='cpu'
)
model = GPT(GPTConfig())
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_scores = test_induction_score(model, seq_len_half=50)

In [ ]:
class SparseAutoEncoder(nn.Module):
    def __init__(self, d_model = 768, n_features = 8192):
        super().__init__()
        self.encoder = nn.Linear(d_model, n_features)
        self.decoder = nn.Linear(n_features, d_model, bias=False)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        x_centered = x - self.pre_bias
        hidden = F.relu(self.encoder(x_centered))
        reconstructed = self.decoder(hidden) + self.pre_bias
        return reconstructed, hidden

    def loss(self, x, reconstructed, hidden, l1_coeff=0.01):
        mse = F.mse_loss(reconstructed, x)
        l1 = hidden.abs().mean()
        total = mse + l1_coeff * l1
        return total, mse, l1

@torch.no_grad()
def collect_activations(model, dataloader, layer_idx=6, n_batches=100):
    all_activations = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)
    model.eval()
    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        model = model.to(device)
        model(x)
        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])
        all_activations.append(acts)

    handle.remove()
    return torch.cat(all_activations, dim=0)

def train_sae(activations, n_epochs=5, batch_size=4096, lr=3e-4, l1_coeff=0.01):
    sae = SparseAutoEncoder(d_model=768, n_features=8192).to(device)

    optimizer = torch.optim.AdamW(sae.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(activations)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(n_epochs):
      all_hidden = []
      total_loss = 0
      total_mse = 0
      n_batches = 0
      for (batch,) in loader:
          batch = batch.to(device)
          reconstructed, hidden = sae(batch)
          total, mse, l1 = sae.loss(batch, reconstructed, hidden, l1_coeff)

          optimizer.zero_grad()
          total.backward()
          optimizer.step()

          all_hidden.append(hidden.detach().cpu())
          total_loss += total.item()
          total_mse += mse.item()
          n_batches += 1

      all_hidden = torch.cat(all_hidden, dim=0)

      avg_active = ((all_hidden > 0).float().sum(dim=1)).mean()
      dead_features = (all_hidden.max(dim=0).values == 0).sum()

      print(f"Epoch {epoch+1}/{n_epochs} | "
            f"Loss: {total_loss/n_batches:.4f} | "
            f"MSE: {total_mse/n_batches:.4f} | "
            f"Active: {avg_active:.1f} | "
            f"Dead: {dead_features}/8192")

    return sae

@torch.no_grad()
def find_top_activating(sae, model, dataloader, feature_id,
                         n_batches=50, top_k=10, layer_idx=6):
    model.eval()
    results = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)

    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        B, T = x.shape

        model(x)

        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])

        reconstructed, hidden = sae(acts)
        score = hidden[:, feature_id]

        for pos in range(len(score)):
            if score[pos] > 0:
                b_idx = pos // T
                t_idx = pos % T

                context_start = max(0, t_idx - 5)
                context_end = min(T, t_idx + 3)
                context = x[b_idx, context_start:context_end].tolist()

                results.append({
                    'score': score[pos].item(),
                    'token_pos': t_idx,
                    'context': context
                })

    handle.remove()
    results.sort(key=lambda r: r['score'], reverse=True)
    return results[:top_k]

In [ ]:
import numpy as np
import os
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GPT(GPTConfig())
model = model.to(device)


project_dir = '*'

class DataLoaderLite:
    def __init__(self, B, T, split='train'):
        self.B = B
        self.T = T
        filename = os.path.join(project_dir, f'{split}.bin')
        self.tokens = np.memmap(filename, dtype=np.uint16, mode='r')
        print(f"[{split}] Загружено {len(self.tokens):,} токенов")
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        buf = torch.tensor(buf.astype(np.int64), dtype=torch.long)
        x = (buf[:-1]).view(B, T)
        y = (buf[1:]).view(B, T)
        self.current_position += B * T
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0
        return x, y

train_loader = DataLoaderLite(B=2, T=512, split='train')
acts = (collect_activations(model, train_loader, layer_idx=6, n_batches=50))
sae = train_sae(acts, n_epochs=5, batch_size=4096, l1_coeff=3.9)

In [ ]:
!find /content/drive/MyDrive -name "*.bin" 2>/dev/null

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')

for feature_id in range(100):
    results = find_top_activating(
        sae, model, train_loader,
        feature_id, n_batches=20
    )

    if results and results[0]['score'] > 1.0:
        print(f"=== Feature #{feature_id} (max score: {results[0]['score']:.2f}) ===")
        for r in results[:3]:
            try:
                text = enc.decode(r['context'])
                print(f"  Score: {r['score']:.2f} | {repr(text)}")
            except:
                print(f"  Score: {r['score']:.2f} | tokens: {r['context']}")
        print()

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')

train_loader = DataLoaderLite(B=2, T=512, split='train')

for feature_id in range(100):
    results = find_top_activating(
        sae, model, train_loader,
        feature_id, n_batches=20
    )

    if results and results[0]['score'] > 1.0:
        print(f"=== Feature #{feature_id} (max score: {results[0]['score']:.2f}) ===")
        for r in results[:3]:
            try:
                text = enc.decode(r['context'])
                print(f"  Score: {r['score']:.2f} | {repr(text)}")
            except:
                print(f"  Score: {r['score']:.2f} | tokens: {r['context']}")
        print()

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')

train_loader = DataLoaderLite(B=2, T=512, split='train')

for feature_id in range(100):
    results = find_top_activating(
        sae, model, train_loader,
        feature_id, n_batches=20
    )

    if results and results[0]['score'] > 0.1:
        print(f"=== Feature #{feature_id} (max score: {results[0]['score']:.2f}) ===")
        for r in results[:3]:
            try:
                text = enc.decode(r['context'])
                print(f"  Score: {r['score']:.2f} | {repr(text)}")
            except:
                print(f"  Score: {r['score']:.2f} | tokens: {r['context']}")
        print()

In [ ]:
test_texts = [
    "up and up and up and up",
    "the cat sat the cat sat the cat",
    "Anna saw Anna and Anna said"
]

for text in test_texts:
    tokens = enc.encode(text)
    x = torch.tensor([tokens]).to(device)

    acts_test = {}
    def hook(m, i, o): acts_test['l'] = o
    h = model.transformer.h[6].register_forward_hook(hook)
    model(x)
    h.remove()

    acts_vec = acts_test['l'].detach().view(-1, 768)
    _, hidden = sae(acts_vec)

    scores_80 = hidden[:, 80].tolist()
    tokens_text = [enc.decode([t]) for t in tokens]

    print(f"\nТекст: {text}")
    for tok, score in zip(tokens_text, scores_80):
        if score > 0.05:
            print(f"  '{tok}': {score:.3f}")

In [ ]:
test_text = "The cat sat on the mat. The cat sat on"
tokens = enc.encode(test_text)
x = torch.tensor([tokens]).to(device)

acts_test = {}
def hook(m, i, o): acts_test['l'] = o
h = model.transformer.h[6].register_forward_hook(hook)
model(x)
h.remove()

acts_vec = acts_test['l'].detach().view(-1, 768)
_, hidden = sae(acts_vec)

tokens_text = [enc.decode([t]) for t in tokens]

print(f"Текст: {test_text}\n")
for pos, (tok, h_vec) in enumerate(zip(tokens_text, hidden)):
    top_features = h_vec.topk(3)
    top_ids = top_features.indices.tolist()
    top_vals = top_features.values.tolist()
    if max(top_vals) > 0.05:
        print(f"  Pos {pos:2d} '{tok}': "
              f"F#{top_ids[0]}={top_vals[0]:.3f} "
              f"F#{top_ids[1]}={top_vals[1]:.3f} "
              f"F#{top_ids[2]}={top_vals[2]:.3f}")

In [ ]:
test_text = "The cat sat on the mat. The cat sat on"
tokens = enc.encode(test_text)
x = torch.tensor([tokens]).to(device)

acts_test = {}
def hook(m, i, o): acts_test['l'] = o
h = model.transformer.h[6].register_forward_hook(hook)
model(x)
h.remove()

acts_vec = acts_test['l'].detach().view(-1, 768)
_, hidden = sae(acts_vec)

feature_id = 8107
scores = hidden[:, feature_id].tolist()
tokens_text = [enc.decode([t]) for t in tokens]

print("*")
print(f"{'*':>4} {'*':>10} {'*':>8}")
print("-" * 30)
for pos, (tok, score) in enumerate(zip(tokens_text, scores)):
    marker = "*" if pos >= 7 else ""
    print(f"{pos:>4} {repr(tok):>10} {score:>8.4f}{marker}")

print(f"*': {scores[1]:.4f}")
print(f"*': {scores[8]:.4f}")
print(f"*: {scores[8] - scores[1]:.4f}")

print(f"*' sat': {scores[2]:.4f}")
print(f"*': {scores[9]:.4f}")
print(f"*: {scores[9] - scores[2]:.4f}")

In [ ]:
test_text = "The cat saw the cat"
tokens = enc.encode(test_text)
x = torch.tensor([tokens]).to(device)

acts_test = {}
def hook(m, i, o): acts_test['l'] = o
h = model.transformer.h[6].register_forward_hook(hook)
model(x)
h.remove()

acts_vec = acts_test['l'].detach().view(-1, 768)
_, hidden = sae(acts_vec)

tokens_text = [enc.decode([t]) for t in tokens]

for feat_id in range(8192):
    scores = hidden[:, feat_id].tolist()

    first_half_max = max(scores[:2])
    second_half_max = max(scores[2:])

    if first_half_max < 0.05 and second_half_max > 0.2:
        print(f"Feature #{feat_id}:")
        for pos, (tok, sc) in enumerate(zip(tokens_text, scores)):
            print(f"  Pos {pos} '{tok}': {sc:.4f}")
        print()

In [ ]:
test_texts = [
    "The cat sat on the mat. The cat",
    "Anna went home. Then Anna",
    "Once upon a time. Once upon",
    "hello world hello world hello"
]

for text in test_texts:
    tokens = enc.encode(text)
    x = torch.tensor([tokens]).to(device)

    acts_test = {}
    def hook(m, i, o): acts_test['l'] = o
    h = model.transformer.h[6].register_forward_hook(hook)
    model(x)
    h.remove()

    acts_vec = acts_test['l'].detach().view(-1, 768)
    _, hidden = sae(acts_vec)

    tokens_text = [enc.decode([t]) for t in tokens]

    print(f"Текст: '{text}'")
    for pos, (tok, h_vec) in enumerate(zip(tokens_text, hidden)):
        s635 = hidden[pos, 635].item()
        s765 = hidden[pos, 765].item()
        if s635 > 0.05 or s765 > 0.05:
            print(f"  Pos {pos:2d} '{tok}': F#635={s635:.3f} F#765={s765:.3f}")
    print()

In [ ]:
steps = [14000, 15000, 16000, 17000, 19000]
ih_scores = [0.0365, 0.0529, 0.0537, 0.0410, 0.0425]

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(steps, ih_scores, 'r-o', label='Induction Score (L6H4)', linewidth=2)
ax1.set_xlabel('Step')
ax1.set_ylabel('Induction Score', color='red')
ax1.tick_params(axis='y', labelcolor='red')


plt.title('Induction Head Formation — GPT-2 124M on TinyStories')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.grid(True, alpha=0.3)

plt.savefig('/content/drive/MyDrive/AI_MegaProject/Induction Head — GPT-2 124M on TinyStories.png', dpi=300)
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def plot_feature_activations(model, sae, text, feature_ids, layer=6):
    tokens = tokenizer.encode(text)
    tokens_tensor = torch.tensor([tokens]).to(device)

    activations = {}
    def hook_fn(module, input, output):
        activations['resid'] = output[0].detach()

    handle = model.transformer.h[layer].register_forward_hook(hook_fn)

    with torch.no_grad():
        model(tokens_tensor)

    handle.remove()

    resid = activations['resid'].squeeze(0)
    with torch.no_grad():
        hidden = sae.encode(resid)

    token_labels = [tokenizer.decode([t]) for t in tokens]

    fig, axes = plt.subplots(len(feature_ids), 1, figsize=(max(12, len(tokens) * 0.4), 4 * len(feature_ids)))

    if len(feature_ids) == 1:
        axes = [axes]

    for ax, fid in zip(axes, feature_ids):
        values = hidden[:, fid].cpu().numpy()
        ax.bar(range(len(token_labels)), values, color='steelblue')
        ax.set_xticks(range(len(token_labels)))
        ax.set_xticklabels(token_labels, rotation=45, ha='right', fontsize=8)
        ax.set_title(f'Feature #{fid} activations')
        ax.set_ylabel('Activation')

    plt.tight_layout()
    plt.savefig('sae_features.png', dpi=150, bbox_inches='tight')
    plt.show()

text = "Once upon a time there was a little girl. The little girl liked to play."
plot_feature_activations(model, sae, text, [8107, 635], layer=6)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tokens_8107 = ['The', ' cat', ' sat', ' on', ' the', ' mat', '.', ' The', ' cat', ' sat', ' on']
scores_8107 = [0.2837, 0.5856, 0.8236, 0.4573, 0.7075, 0.6876, 0.8593, 0.7062, 0.7014, 0.8618, 0.8568]
is_repeat_8107 = [False, False, False, False, False, False, False, True, True, True, True]

tokens_635 = [' on', ' the', '.', ' Then']
scores_635 = [0.137, 0.278, 0.174, 0.269]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

colors_8107 = ['#e74c3c' if r else '#3498db' for r in is_repeat_8107]
bars = axes[0].bar(range(len(tokens_8107)), scores_8107, color=colors_8107)
axes[0].set_xticks(range(len(tokens_8107)))
axes[0].set_xticklabels(tokens_8107, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Feature #8107 — активация по токенам\n(синий = первое появление, красный = повтор)', fontsize=11)
axes[0].set_ylabel('Activation score')
axes[0].set_ylim(0, 1.0)

first_mean = np.mean([s for s, r in zip(scores_8107, is_repeat_8107) if not r])
repeat_mean = np.mean([s for s, r in zip(scores_8107, is_repeat_8107) if r])
axes[0].axhline(first_mean, color='#3498db', linestyle='--', alpha=0.5, label=f'Среднее первых: {first_mean:.3f}')
axes[0].axhline(repeat_mean, color='#e74c3c', linestyle='--', alpha=0.5, label=f'Среднее повторов: {repeat_mean:.3f}')
axes[0].legend(fontsize=9)

axes[1].bar(range(len(tokens_635)), scores_635, color='#2ecc71')
axes[1].set_xticks(range(len(tokens_635)))
axes[1].set_xticklabels(tokens_635, rotation=45, ha='right', fontsize=9)
axes[1].set_title('*', fontsize=11)
axes[1].set_ylabel('Activation score')
axes[1].set_ylim(0, 0.5)

plt.tight_layout()
plt.show()